In [1]:
import os
import cv2
import numpy as np
import json
from collections import defaultdict
from ultralytics import YOLO
from IPython.display import Video, display

In [2]:
video_path = "../data/video_mp4/fish_video8.mp4"

In [3]:
output_det_dir = "../data/output/detection"
output_nlp_dir = "../data/output/nlp_payload"


In [4]:
video_name = os.path.splitext(os.path.basename(video_path))[0]
output_video_path = os.path.join(output_det_dir, f"{video_name}_yolo.mp4")
output_json_path = os.path.join(output_nlp_dir, f"{video_name}_nlp.json")

In [5]:
model = YOLO('../data/models/best_fish_yolo.pt')
cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

In [6]:
fourcc = cv2.VideoWriter_fourcc(*'avc1')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

In [7]:
class_stats = defaultdict(lambda: {"count": 0, "max_conf": 0.0, "best_bbox": [], "best_image_bg": None})

In [8]:
frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
   
    #results = model.predict(frame, conf=0.35, iou=0.5, agnostic_nms=True, verbose=False)
    results = model.track(frame, conf=0.45, iou=0.5, agnostic_nms=True, tracker="botsort.yaml", persist=True, verbose=False)
    black_bg_video = np.zeros_like(frame) 
    
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cls_name = model.names[int(box.cls[0])]
        
        class_stats[cls_name]["count"] += 1
        
        if conf > class_stats[cls_name]["max_conf"]:
            class_stats[cls_name]["max_conf"] = conf
            class_stats[cls_name]["best_bbox"] = [x1, y1, x2, y2]
            
            individual_black_bg = np.zeros_like(frame)
            individual_black_bg[y1:y2, x1:x2] = frame[y1:y2, x1:x2]
            class_stats[cls_name]["best_image_bg"] = individual_black_bg
            
        black_bg_video[y1:y2, x1:x2] = frame[y1:y2, x1:x2]
        cv2.rectangle(black_bg_video, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(black_bg_video, f"{cls_name} {conf:.2f}", (x1, max(10, y1-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                    
    out.write(black_bg_video)
    frame_idx += 1

cap.release()
out.release()
print(f"видео сохранено: {output_video_path}")

видео сохранено: ../data/output/detection/fish_video8_yolo.mp4


In [9]:
threshold_frames = total_frames * 0.1 
final_nlp_data = {"detected_objects": []}

for cls_name, stats in class_stats.items():
    if stats["count"] >= threshold_frames:
        img_name = f"{video_name}_{cls_name}_best.jpg"
        saved_image_path = os.path.join(output_nlp_dir, img_name)
        cv2.imwrite(saved_image_path, stats["best_image_bg"])
        
        final_nlp_data["detected_objects"].append({
            "class": cls_name,
            "confidence": round(stats["max_conf"], 2),
            "bbox": stats["best_bbox"],
            "frames_present": stats["count"],
            "image_path": saved_image_path 
        })

In [10]:
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(final_nlp_data, f, indent=4, ensure_ascii=False)

In [11]:
display(Video(output_video_path, embed=True, width=640))